<a href="https://colab.research.google.com/github/kunjana30/Internship-Project-2/blob/main/Task1_Data_Cleaning_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1 (Easy) – Netflix Data Cleaning & Preparation

**Goal:** Prepare and organise the Netflix dataset so it is ready for business analytics and reporting.

**What this notebook produces:** a clean file called `netflix_cleaned.csv` that Tasks 2–6 load as their starting point.
**Run this notebook first.**

| Step | What we do |
|------|-----------|
| 1 | Import the dataset using Python and Pandas |
| 2 | Identify and handle missing values |
| 3 | Remove duplicate records and formatting inconsistencies |
| 4 | Standardise columns such as Country, Rating and Type |
| 5 | Export the cleaned dataset for analysis |

**Before you start**
* **Google Colab:** upload `Dataset.csv` to a Drive folder named `Netflix_Analysis` (`MyDrive/Netflix_Analysis/`). The setup cell below connects to that folder.
* **Local Jupyter:** put `Dataset.csv` in the *same folder* as this notebook.

### ☁️ Google Colab / Drive setup – run this cell first
Colab's storage is **temporary**: files in `/content` disappear when the session ends, and each notebook runs in its own session.
So this cell connects to your **Google Drive** and works inside a folder called `Netflix_Analysis`, keeping `Dataset.csv`,
`netflix_cleaned.csv` (created by Task 1) and all outputs in one permanent place.

* A pop-up will ask you to allow Drive access - choose your account and click **Allow**.
* Make sure `MyDrive/Netflix_Analysis/` contains `Dataset.csv` (and `netflix_cleaned.csv` once Task 1 has been run).
* **Running locally in Jupyter?** This cell does nothing outside Colab, so you can leave it as it is.
* **No Drive?** Set `USE_DRIVE = False` and upload the files with the Files panel (folder icon on the left). They are deleted when the runtime resets.

In [ ]:
import os, sys

USE_DRIVE = True                                          # set to False to skip Google Drive
PROJECT_DIR = "/content/drive/MyDrive/Netflix_Analysis"   # the folder in your Drive that holds the project files

if "google.colab" in sys.modules and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)

print("Working folder:", os.getcwd())
print("Files here    :", sorted(os.listdir(".")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder: /content/drive/MyDrive/Netflix_Analysis
Files here    : ['Dataset.csv', 'Task1_Data_Cleaning_Preparation.ipynb', 'Task2_Content_Type_Analysis.ipynb', 'Task3_Country_Wise_Analysis.ipynb', 'Task4_Trend_Analysis_Release_Year.ipynb', 'Task5_Content_Rating_Genre_Analysis.ipynb', 'Task6_Business_Insights_Report.ipynb', 'outputs']


---
## Step 1 – Import the dataset using Python and Pandas

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW_PATH = Path("Dataset.csv")            # input  (raw data)
CLEAN_PATH = Path("netflix_cleaned.csv")  # output (cleaned data)
OUT_DIR = Path("outputs/task1_data_cleaning")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(f"'{RAW_PATH}' not found in {Path.cwd()}. Upload Dataset.csv to your Drive folder (Colab) or next to this notebook (local).")

df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()      # work on a copy so the raw data is never modified

# A running log of every cleaning action - handy for your report later
cleaning_log = []
def log(step, action, rows_affected):
    cleaning_log.append({"step": step, "action": action, "rows_affected": int(rows_affected)})

print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head()

Rows: 8,790  |  Columns: 10


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [ ]:
# Structure of the data: data types, non-null counts, unique values and an example of each column
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "unique_values": df.nunique(),
    "example": df.iloc[0],
})
profile

,dtype,non_null,unique_values,example
show_id,object,8790,8790,s1
type,object,8790,2,Movie
title,object,8790,8787,Dick Johnson Is Dead
director,object,8790,4528,Kirsten Johnson
country,object,8790,86,United States
date_added,object,8790,1713,9/25/2021
release_year,int64,8790,74,2020
rating,object,8790,14,PG-13
duration,object,8790,220,90 min
listed_in,object,8790,513,Documentaries


**First observations**

| Column | Meaning | Issue spotted |
|---|---|---|
| `show_id` | Unique ID of the title | – |
| `type` | Movie or TV Show | check spelling/case |
| `title` | Name of the title | check stray spaces |
| `director` | Director(s) | many values look like placeholders |
| `country` | Country of production | placeholders + needs standardising |
| `date_added` | Date added to Netflix | stored as **text**, not a date |
| `release_year` | Year of release | – |
| `rating` | Maturity rating (TV-MA, PG-13 …) | some near-duplicate categories |
| `duration` | `"90 min"` or `"2 Seasons"` | **two units mixed in one column** |
| `listed_in` | Genres, comma-separated | multiple values per cell |

---
## Step 2 – Identify and handle missing values

Start with the standard check: how many `NaN` values does each column have?

In [ ]:
missing = df.isna().sum().to_frame("missing_count")
missing["missing_%"] = (missing["missing_count"] / len(df) * 100).round(2)
missing

,missing_count,missing_%
show_id,0,0.0
type,0,0.0
title,0,0.0
director,0,0.0
country,0,0.0
date_added,0,0.0
release_year,0,0.0
rating,0,0.0
duration,0,0.0
listed_in,0,0.0


Pandas reports **zero** missing values. Don't trust that yet. Missing data is often *disguised* as text such as
`"Not Given"`, `"Unknown"` or `"N/A"`, which pandas does not recognise as `NaN`. Let's scan every text column for those placeholders.

In [ ]:
TEXT_COLS = ["show_id", "type", "title", "director", "country", "date_added", "rating", "duration", "listed_in"]
PLACEHOLDERS = {"not given", "unknown", "n/a", "na", "none", "null", "nan", "-", "?", ""}

def placeholder_report(frame):
    rows = []
    for col in TEXT_COLS:
        is_ph = frame[col].astype(str).str.strip().str.lower().isin(PLACEHOLDERS)
        if is_ph.any():
            rows.append({
                "column": col,
                "placeholder_rows": int(is_ph.sum()),
                "% of rows": round(is_ph.mean() * 100, 2),
                "values_found": sorted(frame.loc[is_ph, col].unique().tolist()),
            })
    return pd.DataFrame(rows)

placeholder_report(df)

,column,placeholder_rows,% of rows,values_found
0,title,1,0.01,[Unknown]
1,director,2588,29.44,[Not Given]
2,country,287,3.27,[Not Given]


In [ ]:
# The 'title' hit looks suspicious - is it really missing data?
df[df["title"].str.lower() == "unknown"]

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
1336,s1468,Movie,Unknown,Jaume Collet-Serra,United Kingdom,1/1/2021,2011,PG-13,113 min,"Action & Adventure, Thrillers"


**Decision table**

| Column | Finding | Decision | Why |
|---|---|---|---|
| `director` | ~29% are `"Not Given"` | Convert to real `NaN`, then fill with `"Unknown"` | Dropping 29% of rows would destroy most of the dataset; director isn't needed for most analyses |
| `country` | ~3% are `"Not Given"` | Convert to `NaN`, then fill with `"Unknown"` | Keep the rows for type/year/rating analysis; Task 3 excludes `"Unknown"` from country rankings |
| `title` | One title is literally called **"Unknown"** | **Leave as is** | It is a *real movie* (Jaume Collet-Serra, 2011) – a blind find-and-replace would corrupt valid data |

Lesson: always inspect before replacing. Only `director` and `country` get the placeholder treatment.

In [ ]:
PLACEHOLDER_COLS = ["director", "country"]

for col in PLACEHOLDER_COLS:
    is_ph = df[col].str.strip().str.lower().isin(PLACEHOLDERS)
    df.loc[is_ph, col] = np.nan                  # 1) make the missing values visible to pandas
    log("2. Missing values", f"'{col}': placeholder text -> NaN -> 'Unknown'", is_ph.sum())

print("Real NaN values now visible to pandas:")
print(df[PLACEHOLDER_COLS].isna().sum())

for col in PLACEHOLDER_COLS:
    df[col] = df[col].fillna("Unknown")           # 2) fill with one consistent label

assert df.isna().sum().sum() == 0, "There are still missing values!"
print("\nMissing values after handling:", int(df.isna().sum().sum()))

Real NaN values now visible to pandas:
director    2588
country      287
dtype: int64

Missing values after handling: 0


---
## Step 3 – Remove duplicate records and formatting inconsistencies

### 3a. Fix whitespace
Leading/trailing/double spaces make identical values look different (`"Consequences "` ≠ `"Consequences"`).

In [ ]:
before = df[TEXT_COLS].copy()

for col in TEXT_COLS:
    df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

changed = (before != df[TEXT_COLS]).sum()
changed = changed[changed > 0]
print("Cells fixed per column:")
print(changed if len(changed) else "none")
log("3. Formatting", "Stripped leading/trailing/double spaces in text columns", changed.sum())

Cells fixed per column:
title    9
dtype: int64


### 3b. Find and remove duplicates
There are three levels of "duplicate". Check them all:
1. Exact duplicate rows
2. Duplicate `show_id` values (the ID should be unique)
3. **Hidden duplicates** – the same title appearing under a *different* `show_id`

In [ ]:
content_cols = [c for c in df.columns if c != "show_id"]   # everything except the ID

print("1) Exact duplicate rows              :", df.duplicated().sum())
print("2) Duplicate show_id values          :", df["show_id"].duplicated().sum())
print("3) Same content, different show_id   :", df.duplicated(subset=content_cols).sum())

hidden = df[df.duplicated(subset=content_cols, keep=False)].sort_values(["title", "show_id"])
hidden

1) Exact duplicate rows              : 0
2) Duplicate show_id values          : 0
3) Same content, different show_id   : 4


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
2925,s3963,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
3285,s4523,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
4260,s5966,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
393,s3997,TV Show,9-Feb,Unknown,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
537,s5965,TV Show,9-Feb,Unknown,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
2590,s3372,Movie,Consequences,Ozan Açıktan,Turkey,10/25/2019,2014,TV-MA,106 min,"Dramas, International Movies, Thrillers"
4718,s6530,Movie,Consequences,Ozan Açıktan,Turkey,10/25/2019,2014,TV-MA,106 min,"Dramas, International Movies, Thrillers"


Level 1 and 2 checks pass, but level 3 finds **4 titles stored twice** (identical in every column except `show_id`).
Notice that one of them (*Consequences*) only matches **because we fixed the whitespace in 3a first**, so the order of cleaning steps matters.
These are true duplicates, so we keep the first occurrence and drop the rest.

In [ ]:
n_before = len(df)
df = df.drop_duplicates(subset=content_cols, keep="first").reset_index(drop=True)
removed = n_before - len(df)
log("3. Duplicates", "Dropped records identical in every column except show_id", removed)
print(f"Removed {removed} duplicate rows  ->  {len(df):,} rows remain")

Removed 4 duplicate rows  ->  8,786 rows remain


In [ ]:
# Near-duplicates: same title (ignoring capitalisation), type, release year and country - but NOT identical rows
tmp = df.assign(title_key=df["title"].str.lower())
near = tmp[tmp.duplicated(subset=["title_key", "type", "release_year", "country"], keep=False)]
near = near.sort_values(["title_key", "date_added"])
print(f"{len(near)} rows ({len(near) // 2} pairs) look like the same production listed twice, e.g. 'Love in a Puff' vs 'Love In A Puff'")
near[["show_id", "title", "type", "release_year", "country", "date_added", "rating"]]

6 rows (3 pairs) look like the same production listed twice, e.g. 'Love in a Puff' vs 'Love In A Puff'


,show_id,title,type,release_year,country,date_added,rating
4852,s6706,Esperando La Carroza,Movie,1985,Argentina,7/15/2018,NR
415,s304,Esperando la carroza,Movie,1985,Argentina,8/5/2021,TV-MA
5357,s7346,Love In A Puff,Movie,2010,Hong Kong,8/1/2018,TV-MA
220,s160,Love in a Puff,Movie,2010,Hong Kong,9/1/2021,TV-MA
8663,s8023,Sin Senos sí Hay Paraíso,TV Show,2018,United States,1/11/2019,TV-MA
6850,s1271,Sin senos sí hay paraíso,TV Show,2018,United States,2/25/2021,TV-MA


**Decision on near-duplicates:** the pairs above differ in `date_added` (and sometimes `rating`), so they may be genuine re-listings
and we cannot be certain they are errors. They are only ~0.03% of the data, so we **keep and flag them** rather than guess.
(Note that titles like *Death Note* – a 2006 TV Show and a 2017 Movie – are different productions and were correctly excluded
because the `type` and `release_year` differ.)

### 3c. Fix data types – convert `date_added` from text to a real date
Real dates let us extract year/month and sort chronologically.

In [ ]:
df["date_added"] = pd.to_datetime(df["date_added"], format="%m/%d/%Y", errors="coerce")
bad_dates = df["date_added"].isna().sum()
assert bad_dates == 0, f"{bad_dates} dates could not be parsed - check the format"
log("3. Formatting", "date_added: text -> datetime", len(df))

# helpful derived columns for the time-based analysis in Tasks 4 & 6
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month
df["month_name"] = df["date_added"].dt.month_name()

print("date_added range :", df["date_added"].min().date(), "->", df["date_added"].max().date())
print("release_year range:", df["release_year"].min(), "->", df["release_year"].max())

# sanity check: titles added to Netflix BEFORE their listed release year
early = df[df["year_added"] < df["release_year"]]
print(f"\n{len(early)} titles have year_added < release_year (e.g. added ahead of release) -> kept, but good to know")
early[["title", "type", "date_added", "release_year"]].head()

date_added range : 2008-01-01 -> 2021-09-25
release_year range: 1925 -> 2021

14 titles have year_added < release_year (e.g. added ahead of release) -> kept, but good to know


,title,type,date_added,release_year
3868,Hans Teeuwen: Real Rancour,Movie,2017-07-01,2018
5134,Incoming,Movie,2018-10-26,2019
5174,Jack Taylor,TV Show,2013-03-31,2016
6944,Hilda,TV Show,2020-12-14,2021
6988,Polly Pocket,TV Show,2020-11-15,2021


---
## Step 4 – Standardise columns (Type, Country, Rating, Duration, Genres)

### 4a. `type`

In [ ]:
print("Before:", sorted(df["type"].unique()))

TYPE_MAP = {"movie": "Movie", "movies": "Movie", "tv show": "TV Show", "tv shows": "TV Show", "tv series": "TV Show"}
df["type"] = df["type"].str.lower().map(TYPE_MAP)
assert df["type"].notna().all(), "Unexpected value in 'type' - add it to TYPE_MAP"

print("After :", sorted(df["type"].unique()))
df["type"].value_counts()

Before: ['Movie', 'TV Show']
After : ['Movie', 'TV Show']


,count
type,
Movie,6123
TV Show,2663


### 4b. `country`
Consistent capitalisation, and merge obsolete names. In this file each title has **one** country (no comma-separated lists) – we verify that rather than assume it.

In [ ]:
print("Rows with multiple countries (comma-separated):", df["country"].str.contains(",").sum())

old_country = df["country"].copy()
df["country"] = df["country"].str.title()

# 'West Germany' no longer exists - treat as Germany so it isn't counted as a separate country.
# (We keep 'Soviet Union' as it is, because it is not equal to modern Russia.)
COUNTRY_FIXES = {"West Germany": "Germany"}
df["country"] = df["country"].replace(COUNTRY_FIXES)

log("4. Standardise", "country: title-case + merged obsolete names", (old_country != df["country"]).sum())
print(f"Distinct countries (incl. 'Unknown'): {df['country'].nunique()}")
df["country"].value_counts().head(10)

Rows with multiple countries (comma-separated): 0
Distinct countries (incl. 'Unknown'): 85


,count
country,
United States,3240
India,1056
United Kingdom,638
Pakistan,420
Unknown,287
Canada,271
Japan,259
South Korea,214
France,213


### 4c. `rating`
* `NR` (Not Rated) and `UR` (Unrated) mean the same thing → merge into `NR`.
* Add a `rating_group` column that translates 14 rating codes into 4 easy-to-read audience groups – this will power the audience analysis in Tasks 5 and 6.

| Audience group | Ratings |
|---|---|
| Kids | TV-Y, TV-G, G |
| Older Kids | TV-Y7, TV-Y7-FV, TV-PG, PG |
| Teens | PG-13, TV-14 |
| Adults | R, TV-MA, NC-17 |
| Not Rated | NR |

*(Grouping is a judgement call; document it in your report.)*

In [ ]:
print("Before:", sorted(df["rating"].unique()))

df["rating"] = df["rating"].str.upper()
n_ur = (df["rating"] == "UR").sum()
df["rating"] = df["rating"].replace({"UR": "NR"})
log("4. Standardise", "rating: merged 'UR' into 'NR'", n_ur)

RATING_GROUP = {
    "TV-Y": "Kids", "TV-G": "Kids", "G": "Kids",
    "TV-Y7": "Older Kids", "TV-Y7-FV": "Older Kids", "TV-PG": "Older Kids", "PG": "Older Kids",
    "PG-13": "Teens", "TV-14": "Teens",
    "R": "Adults", "TV-MA": "Adults", "NC-17": "Adults",
    "NR": "Not Rated",
}
df["rating_group"] = df["rating"].map(RATING_GROUP)
assert df["rating_group"].notna().all(), "A rating is missing from RATING_GROUP"

print("After :", sorted(df["rating"].unique()))
df["rating_group"].value_counts()

Before: ['G', 'NC-17', 'NR', 'PG', 'PG-13', 'R', 'TV-14', 'TV-G', 'TV-MA', 'TV-PG', 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'UR']
After : ['G', 'NC-17', 'NR', 'PG', 'PG-13', 'R', 'TV-14', 'TV-G', 'TV-MA', 'TV-PG', 'TV-Y', 'TV-Y7', 'TV-Y7-FV']


,count
rating_group,
Adults,4005
Teens,2645
Older Kids,1487
Kids,567
Not Rated,82


### 4d. `duration`
This column mixes minutes (movies) and seasons (TV shows), so it can't be averaged or sorted as-is.
We split it into two numeric columns: `duration_min` (movies) and `seasons` (TV shows).

In [ ]:
parts = df["duration"].str.extract(r"(?P<value>\d+)\s*(?P<unit>min|Seasons?)")
assert parts.notna().all().all(), "Some 'duration' values did not match the expected pattern"

number = parts["value"].astype(int)
df["duration_min"] = number.where(parts["unit"] == "min").astype("Int64")                  # movies only
df["seasons"] = number.where(parts["unit"].str.startswith("Season")).astype("Int64")      # TV shows only
log("4. Standardise", "duration: split into duration_min / seasons", len(df))

# consistency check: movies should have minutes, TV shows should have seasons
print(pd.crosstab(df["type"], parts["unit"]))
df[["type", "duration", "duration_min", "seasons"]].head()

unit     Season  Seasons   min
type                          
Movie         0        0  6123
TV Show    1790      873     0


,type,duration,duration_min,seasons
0,Movie,90 min,90,<NA>
1,TV Show,1 Season,<NA>,1
2,TV Show,1 Season,<NA>,1
3,Movie,91 min,91,<NA>
4,Movie,125 min,125,<NA>


### 4e. `listed_in` (genres)
Normalise the separator and add two helper columns: `n_genres` (how many genres a title has) and `primary_genre` (the first one listed).

In [ ]:
df["listed_in"] = df["listed_in"].str.replace(r"\s*,\s*", ", ", regex=True)
df["n_genres"] = df["listed_in"].str.count(",") + 1
df["primary_genre"] = df["listed_in"].str.split(",").str[0].str.strip()
log("4. Standardise", "listed_in: normalised separators, added n_genres & primary_genre", len(df))

df[["title", "listed_in", "n_genres", "primary_genre"]].head()

,title,listed_in,n_genres,primary_genre
0,Dick Johnson Is Dead,Documentaries,1,Documentaries
1,Ganglands,"Crime TV Shows, International TV Shows, TV Act...",3,Crime TV Shows
2,Midnight Mass,"TV Dramas, TV Horror, TV Mysteries",3,TV Dramas
3,Confessions of an Invisible Girl,"Children & Family Movies, Comedies",2,Children & Family Movies
4,Sankofa,"Dramas, Independent Movies, International Movies",3,Dramas


---
## Step 5 – Validate and export the cleaned dataset
Before saving, run automated checks. If any `assert` fails, something is wrong and you'll know immediately.

In [ ]:
FINAL_COLUMNS = [
    "show_id", "type", "title", "director", "country",
    "date_added", "year_added", "month_added", "month_name", "release_year",
    "rating", "rating_group", "duration", "duration_min", "seasons",
    "listed_in", "n_genres", "primary_genre",
]
df_clean = df[FINAL_COLUMNS].sort_values("date_added", ascending=False).reset_index(drop=True)

# ---- automated quality checks ----
assert df_clean["show_id"].is_unique,                                  "show_id is not unique"
assert not df_clean.duplicated(subset=[c for c in FINAL_COLUMNS if c != "show_id"]).any(), "duplicates remain"
assert df_clean.isna().sum()[["type", "title", "director", "country", "date_added", "rating"]].sum() == 0, "NaN in key columns"
assert set(df_clean["type"]) == {"Movie", "TV Show"},                  "unexpected type values"
assert df_clean["release_year"].between(1900, 2030).all(),             "release_year out of range"
assert (df_clean["duration_min"].notna() == (df_clean["type"] == "Movie")).all(), "movies/minutes mismatch"
assert (df_clean["seasons"].notna() == (df_clean["type"] == "TV Show")).all(),    "TV shows/seasons mismatch"
print("All quality checks passed ✔")

# ---- before vs after ----
summary = pd.DataFrame({
    "Raw data": [len(df_raw), df_raw.shape[1], int((df_raw["director"] == "Not Given").sum()),
                 int((df_raw["country"] == "Not Given").sum()), df_raw["rating"].nunique(), df_raw["country"].nunique()],
    "Cleaned data": [len(df_clean), df_clean.shape[1], int((df_clean["director"] == "Unknown").sum()),
                     int((df_clean["country"] == "Unknown").sum()), df_clean["rating"].nunique(), df_clean["country"].nunique()],
}, index=["Rows", "Columns", "Director placeholder rows", "Country placeholder rows", "Distinct ratings", "Distinct countries"])
summary

All quality checks passed ✔


,Raw data,Cleaned data
Rows,8790,8786
Columns,10,18
Director placeholder rows,2588,2587
Country placeholder rows,287,287
Distinct ratings,14,13
Distinct countries,86,85


In [ ]:
# The cleaning log - paste this table into your report
log_df = pd.DataFrame(cleaning_log)
log_df

,step,action,rows_affected
0,2. Missing values,'director': placeholder text -> NaN -> 'Unknown',2588
1,2. Missing values,'country': placeholder text -> NaN -> 'Unknown',287
2,3. Formatting,Stripped leading/trailing/double spaces in tex...,9
3,3. Duplicates,Dropped records identical in every column exce...,4
4,3. Formatting,date_added: text -> datetime,8786
5,4. Standardise,country: title-case + merged obsolete names,1
6,4. Standardise,rating: merged 'UR' into 'NR',3
7,4. Standardise,duration: split into duration_min / seasons,8786
8,4. Standardise,"listed_in: normalised separators, added n_genr...",8786


In [ ]:
# ---- export ----
df_clean.to_csv(CLEAN_PATH, index=False)
log_df.to_csv(OUT_DIR / "cleaning_log.csv", index=False)
summary.to_csv(OUT_DIR / "before_vs_after_summary.csv")

# ---- verify the saved file loads correctly ----
check = pd.read_csv(CLEAN_PATH, parse_dates=["date_added"])
assert check.shape == df_clean.shape, "Saved file has a different shape!"
print(f"Saved '{CLEAN_PATH}'  ->  {check.shape[0]:,} rows x {check.shape[1]} columns")
print(f"Cleaning log + summary saved in '{OUT_DIR}/'")
check.dtypes.astype(str).to_frame("dtype")

Saved 'netflix_cleaned.csv'  ->  8,786 rows x 18 columns
Cleaning log + summary saved in 'outputs/task1_data_cleaning/'


,dtype
show_id,object
type,object
title,object
director,object
country,object
date_added,datetime64[ns]
year_added,int64
month_added,int64
month_name,object
release_year,int64


---
## ✅ Summary

* Imported the raw dataset and profiled its structure.
* Discovered **disguised missing values** (`"Not Given"` in `director` and `country`) that `isna()` could not see, and handled them without dropping rows.
* Removed **3 hidden duplicates** and fixed whitespace.
* Converted `date_added` to a real date and split `duration` into numeric columns.
* Standardised `type`, `country` and `rating`, and engineered `rating_group`, `year_added`, `month_added`, `n_genres`, `primary_genre`.
* Validated the result with automated checks and exported `netflix_cleaned.csv`.

**Next:** open Task 2, 3, 4, 5 or 6. Each starts by loading `netflix_cleaned.csv`.